In [1]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui import magicgui
from magicgui.widgets import TextEdit
from napari.qt.threading import thread_worker
import os
import tempfile


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:



def safe_imwrite(path, arr, *, imagej=True, resolution=None, resolutionunit=None,
                 metadata=None):
    """Atomically write a TIFF so a failed write can never corrupt an existing file.

    tifffile.imwrite truncates the destination to 0 bytes the moment it opens it,
    then writes. If the write raises part-way (e.g. imagej=True with an
    unsupported dtype like float64) the original bytes are already gone and the
    file is left unopenable.

    This helper instead writes to a temp file in the SAME folder, verifies the
    pixel data reads back identically, and only then atomically replaces the
    target with os.replace. The original is untouched unless the new file is
    known-good.
    """
    path = os.fspath(path)
    arr = np.asarray(arr)

    # ImageJ TIFF only supports these dtypes -> fail loudly BEFORE touching disk.
    if imagej and arr.dtype not in (np.uint8, np.uint16, np.int16, np.float32):
        raise TypeError(
            f"imagej=True cannot store dtype {arr.dtype}. Cast first, e.g. "
            f"arr.astype(np.float32) (intensities) or arr.astype(np.uint16) (labels)."
        )

    folder = os.path.dirname(path) or "."
    fd, tmp = tempfile.mkstemp(suffix=".tmp.tif", dir=folder)
    os.close(fd)
    try:
        kwargs = {"imagej": imagej}
        if resolution is not None:
            kwargs["resolution"] = resolution
        if resolutionunit is not None:
            kwargs["resolutionunit"] = resolutionunit
        if metadata is not None:
            kwargs["metadata"] = metadata

        tifffile.imwrite(tmp, arr, **kwargs)

        # verify the temp file opens and matches the source before replacing
        if not np.array_equal(tifffile.imread(tmp), arr):
            raise ValueError("verification failed: written data != source array")

        os.replace(tmp, path)  # atomic on the same filesystem
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)
    return path


In [3]:
def read_image_and_meta(path):
    """Read a tif as a (Z, C, Y, X) array along with the metadata needed to
    write it back out unchanged (pixel size / spacing / unit)."""
    with tifffile.TiffFile(path) as tif:
        series = max(tif.series, key=lambda s: s.size)
        arr = series.asarray()
        axes = series.axes
        ij_meta = dict(tif.imagej_metadata or {})

        def _rational(tag):
            if tag is None:
                return None
            val = tag.value
            if isinstance(val, tuple) and len(val) == 2:
                return val[0] / val[1] if val[1] else None
            return val

        xres = yres = resunit = None
        for page in tif.pages:
            if "XResolution" in page.tags:
                xres = _rational(page.tags.get("XResolution"))
                yres = _rational(page.tags.get("YResolution"))
                resunit_tag = page.tags.get("ResolutionUnit")
                resunit = int(resunit_tag.value) if resunit_tag is not None else None
                break

    meta = {
        "axes": axes,
        "imagej": ij_meta,
        "xres": xres,
        "yres": yres,
        "resunit": resunit,
    }
    return arr, meta



In [4]:

def save_image(path, arr, meta):
    """Write `arr` to `path`, preserving the original pixel size / spacing /
    unit metadata.

    The channel / slice / frame counts are recomputed from `arr` (rather than
    copied from the source metadata) so the ImageJ header stays correct even
    when the channel count changes, e.g. when we add a 4th `valid` channel to a
    previously 3-channel stack.

    Uses safe_imwrite so a failed write can never corrupt the existing file.
    """
    ij = meta["imagej"]
    axes = meta["axes"]
    md = {"axes": axes}

    for key in ("spacing", "unit", "finterval", "fps", "mode"):
        if key in ij:
            md[key] = ij[key]

    for ax, name in (("Z", "slices"), ("C", "channels"), ("T", "frames")):
        if ax in axes:
            md[name] = arr.shape[axes.index(ax)]

    kwargs = {"imagej": True, "metadata": md}
    if meta["xres"] and meta["yres"]:
        kwargs["resolution"] = (meta["xres"], meta["yres"])
    if meta["resunit"] is not None:
        kwargs["resolutionunit"] = meta["resunit"]

    safe_imwrite(path, arr, **kwargs)


In [5]:
class MaskCurator:
    """Napari GUI to curate the mask channel of a (Z, C, Y, X) tif.

    Saving strategy (optimised for very large stacks on slow/network drives):
      * While curating, ONLY the mask channel (single channel, ZYX) is autosaved
        to a small `<name>_MASK_AUTOSAVE.tif`, on a BACKGROUND THREAD (napari
        thread_worker) so the GUI stays responsive. The original is not touched.
      * "Save final" COMMITS: it folds the mask into the 3-channel stack, writes
        it OVER the original file, then deletes the now-obsolete `_CURATED` and
        `_MASK_AUTOSAVE` sidecars.
      * Closing the window writes the full 3-channel to a non-destructive
        `<name>_CURATED.tif` safety net -- but ONLY if there are unsaved mask
        edits; if nothing changed since the last save it skips the write.
      * On load it resumes from an existing `_CURATED`, and if a newer
        `_MASK_AUTOSAVE` exists it overlays that onto the mask channel.
    All writes go through safe_imwrite (atomic temp-then-replace), so a failed
    write can never corrupt an existing file.

    Background saves use napari's thread_worker: the worker does the disk write
    off the GUI thread, and its returned/errored/finished signals fire back ON
    the GUI thread, so "save completed" / error messages appear in the Log panel.
    """

    AUTOSAVE_SECONDS = 300

    def __init__(self, default_folder=None,
                 brightfield_channel=0, fluorescence_channel=1, mask_channel=2):
        self.default_folder = Path(default_folder) if default_folder else None
        self.brightfield_channel = brightfield_channel
        self.fluorescence_channel = fluorescence_channel
        self.mask_channel = mask_channel

        self.viewer = None
        self.base_path = None      # original identity used for naming outputs
        self.path = None           # file actually loaded (original or _CURATED)
        self.arr = None
        self.meta = None
        self.mask_layer = None
        self.log_widget = None
        self._orig_close = None
        self._autosave_timer = None
        self._save_worker = None   # currently running background save worker
        self._save_running = False
        self._dirty = False

    def _log(self, message):
        """Append a message to the GUI log panel (falls back to print).

        Safe to call from the GUI thread. Background work is done in
        thread_workers whose signals are delivered on the GUI thread, so their
        callbacks may call this too.
        """
        if self.log_widget is not None:
            current = self.log_widget.value
            self.log_widget.value = (current + "\n" + message) if current else message
        else:
            print(message)

    @staticmethod
    def _rescale_to_255(image):
        """Linearly rescale an image so its min/max map to 0/255 (uint8)."""
        image = image.astype(np.float32)
        lo, hi = float(image.min()), float(image.max())
        if hi > lo:
            image = (image - lo) / (hi - lo) * 255.0
        else:
            image = np.zeros_like(image)
        return image.astype(np.uint8)

    # ------------------------------------------------------------------ paths
    def _curated_path(self):
        """Non-destructive full 3-channel safety net: `<base>_CURATED.tif`."""
        p = self.base_path
        return p.with_name(p.stem + "_CURATED" + p.suffix)

    def _mask_autosave_path(self):
        """Small mask-only autosave: `<base>_MASK_AUTOSAVE.tif`."""
        p = self.base_path
        return p.with_name(p.stem + "_MASK_AUTOSAVE" + p.suffix)

    def _mask_write_kwargs(self):
        """imagej kwargs for writing the mask-only (ZYX) autosave."""
        md = {"axes": "ZYX"}
        ij = self.meta["imagej"]
        for k in ("spacing", "unit"):
            if k in ij:
                md[k] = ij[k]
        kwargs = {"imagej": True, "metadata": md}
        if self.meta["xres"] and self.meta["yres"]:
            kwargs["resolution"] = (self.meta["xres"], self.meta["yres"])
        if self.meta["resunit"] is not None:
            kwargs["resolutionunit"] = self.meta["resunit"]
        return kwargs

    # ------------------------------------------------------------------ GUI
    def start(self):
        from qtpy.QtCore import QTimer

        self.viewer = napari.Viewer(title="Mask curation")

        self.load_widget = magicgui(
            self._load_image,
            image_path={"label": "Image", "mode": "r",
                        "filter": "TIFF (*.tif *.tiff)"},
            call_button="Load image",
        )
        if self.default_folder and self.default_folder.exists():
            self.load_widget.image_path.value = self.default_folder

        # Replace a z-slice (or an inclusive range) of the MASK with another slice.
        self.replace_widget = magicgui(
            self._replace_slices,
            replace_z={"label": "Replace z (e.g. 16 or 1,3)"},
            source_z={"label": "with mask from z"},
            call_button="Replace slices",
        )

        # Manual FINAL save = COMMIT the 3-channel image over the original.
        self.save_widget = magicgui(self._save_full,
                                    call_button="Save final (overwrite original)")

        self.log_widget = TextEdit(value="", label="Log")
        try:
            self.log_widget.native.setReadOnly(True)
        except Exception:
            pass
        self.log_widget.min_height = 120
        self.log_widget.max_height = 300

        self.viewer.window.add_dock_widget(self.load_widget, area="right",
                                           name="Select image")
        self.viewer.window.add_dock_widget(self.replace_widget, area="right",
                                           name="Replace slices")
        self.viewer.window.add_dock_widget(self.save_widget, area="right",
                                           name="Save")
        self.viewer.window.add_dock_widget(self.log_widget, area="right",
                                           name="Log")

        # Autosave (mask only, background thread) on a timer.
        self._autosave_timer = QTimer()
        self._autosave_timer.setInterval(self.AUTOSAVE_SECONDS * 1000)
        self._autosave_timer.timeout.connect(self._autosave)
        self._autosave_timer.start()

        # Finalise (full 3-channel save) when the window is closed.
        qt_window = self.viewer.window._qt_window
        self._orig_close = qt_window.closeEvent
        qt_window.closeEvent = self._on_close

    # ------------------------------------------------------------------ load
    def _load_image(self, image_path=Path()):
        image_path = Path(image_path)
        if not image_path.is_file():
            self._log("[WARN] Please select a tif file.")
            return

        # Identity/base name (strip a _CURATED suffix if the user picked one).
        stem = image_path.stem
        if stem.endswith("_CURATED"):
            self.base_path = image_path.with_name(
                stem[: -len("_CURATED")] + image_path.suffix)
        else:
            self.base_path = image_path

        # Prefer an existing full _CURATED as the source of pixel data.
        curated = self._curated_path()
        source = curated if curated.is_file() else self.base_path
        if source == curated:
            self._log(f"[RESUME] Loading full progress from {curated.name}")

        try:
            arr, meta = read_image_and_meta(source)
        except Exception as e:
            self._log(f"[ERROR] Could not read {source.name}: "
                      f"{type(e).__name__}: {e}")
            return

        self.path = source
        self.arr, self.meta = arr, meta

        # If a newer mask-only autosave exists, overlay it onto the mask channel.
        ma = self._mask_autosave_path()
        if ma.is_file() and ma.stat().st_mtime > source.stat().st_mtime:
            try:
                mask_arr = tifffile.imread(ma)
                expected = self.arr[:, self.mask_channel].shape
                if mask_arr.shape == expected:
                    self.arr[:, self.mask_channel] = mask_arr.astype(self.arr.dtype)
                    self._log(f"[RESUME] Applied newer mask autosave {ma.name}")
                else:
                    self._log(f"[WARN] Mask autosave shape {mask_arr.shape} != "
                              f"expected {expected}; ignored.")
            except Exception as e:
                self._log(f"[WARN] Could not read mask autosave: {e}")

        brightfield = self.arr[:, self.brightfield_channel, :, :]
        fluorescence = self.arr[:, self.fluorescence_channel, :, :]
        mask = self.arr[:, self.mask_channel, :, :]

        # Replace any layers from a previously loaded image.
        self.viewer.layers.clear()
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive", scale=(5, 2, 2))
        self.viewer.add_image(fluorescence, name="fluorescence",
                              colormap="green", blending="additive", scale=(5, 2, 2))
        self.mask_layer = self.viewer.add_labels(mask.astype(np.int32),
                                                 name="mask", scale=(5, 2, 2))
        # Mark unsaved whenever the mask is painted/edited.
        self.mask_layer.events.paint.connect(self._mark_dirty)
        self.mask_layer.events.data.connect(self._mark_dirty)

        self.viewer.layers.selection.active = self.mask_layer
        self.viewer.title = self.base_path.name
        self._dirty = False
        self._log(f"[OK] Loaded {source.name}\n"
                  f"  mask autosave -> {self._mask_autosave_path().name}\n"
                  f"  save final    -> overwrites {self.base_path.name}")

    # ------------------------------------------------------------------ edits
    @staticmethod
    def _parse_z_range(text, nz):
        """'16' -> [16];  '1,3' or '1-3' -> [1,2,3] (inclusive). None if invalid."""
        t = str(text).strip().strip("{}[]()")
        for sep in ("-", " ", ";"):
            t = t.replace(sep, ",")
        parts = [p for p in t.split(",") if p != ""]
        if not parts:
            return None
        try:
            nums = [int(p) for p in parts]
        except ValueError:
            return None
        lo, hi = (nums[0], nums[0]) if len(nums) == 1 else (nums[0], nums[1])
        if lo > hi:
            lo, hi = hi, lo
        if lo < 0 or hi >= nz:
            return None
        return list(range(lo, hi + 1))

    @staticmethod
    def _parse_single_z(text, nz):
        t = str(text).strip().strip("{}[]()")
        try:
            z = int(t)
        except ValueError:
            return None
        return z if 0 <= z < nz else None

    def _replace_slices(self, replace_z: str = "", source_z: str = ""):
        """Copy the MASK from `source_z` into every z in `replace_z`.

        Operates ONLY on the mask layer (channel `mask_channel`); the brightfield
        and fluorescence layers are never touched.
        """
        if self.mask_layer is None:
            self._log("[WARN] Load an image before replacing slices.")
            return

        data = np.asarray(self.mask_layer.data)
        nz = data.shape[0]

        targets = self._parse_z_range(replace_z, nz)
        if targets is None:
            self._log(f"[ERROR] 'Replace z' = {replace_z!r} is invalid "
                      f"(use e.g. 16 or 1,3; valid range 0..{nz - 1}).")
            return
        src = self._parse_single_z(source_z, nz)
        if src is None:
            self._log(f"[ERROR] 'with mask from z' = {source_z!r} is invalid "
                      f"(use a single slice 0..{nz - 1}).")
            return

        src_slice = data[src].copy()
        for z in targets:
            data[z] = src_slice
        self.mask_layer.data = data
        self.mask_layer.refresh()

        self._dirty = True
        if len(targets) == 1:
            self._log(f"[OK] Successfully replaced mask slice {targets[0]} "
                      f"with slice {src}")
        else:
            self._log(f"[OK] Successfully replaced mask slices "
                      f"{targets[0]}-{targets[-1]} with slice {src}")
        self._save_mask_async()  # secure mask progress straight away

    # ------------------------------------------------------------------ save
    def _mark_dirty(self, event=None):
        self._dirty = True

    def _sync_mask_into_arr(self):
        self.arr[:, self.mask_channel, :, :] = self.mask_layer.data.astype(self.arr.dtype)

    def _on_save_finished(self):
        """Runs on the GUI thread when any background save worker ends."""
        self._save_running = False
        self._save_worker = None

    def _save_mask_async(self):
        """Autosave ONLY the mask channel (ZYX) on a background thread."""
        if self.mask_layer is None or self.base_path is None:
            return
        if self._save_running:
            self._log("[autosave] previous save still running; will retry next tick.")
            return

        # Snapshot on the GUI thread so the worker has a stable, private copy.
        mask_snapshot = np.asarray(self.mask_layer.data).astype(self.arr.dtype)
        out = self._mask_autosave_path()
        kwargs = self._mask_write_kwargs()
        self._dirty = False  # captured in snapshot; worker re-flags on failure

        @thread_worker
        def _work():
            safe_imwrite(out, mask_snapshot, **kwargs)
            return out.name

        def _ok(name):
            self._log(f"[autosave] save completed -> {name}")

        def _err(exc):
            self._dirty = True  # force a retry on the next tick
            self._log(f"[autosave][ERROR] {type(exc).__name__}: {exc}")

        worker = _work()
        worker.returned.connect(_ok)
        worker.errored.connect(_err)
        worker.finished.connect(self._on_save_finished)
        self._save_worker = worker
        self._save_running = True
        worker.start()
        self._log(f"[autosave] writing mask in background -> {out.name} ...")

    def _autosave(self):
        if self._dirty and self.mask_layer is not None and self.base_path is not None:
            self._save_mask_async()

    def _save_full(self):
        """FINAL save / COMMIT: fold the mask into the 3-channel array, write it
        OVER the original file, then delete the now-obsolete _CURATED and
        _MASK_AUTOSAVE sidecars. Runs on a background thread.
        """
        if self.mask_layer is None or self.arr is None:
            self._log("[WARN] Load an image before saving.")
            return
        if self._save_running:
            self._log("[SAVE] a save is already in progress; try again shortly.")
            return

        self._sync_mask_into_arr()  # fold current mask into arr on the GUI thread
        self._dirty = False          # snapshot taken; a later edit re-flags via paint
        out = self.base_path                 # COMMIT over the original file
        arr, meta = self.arr, self.meta
        curated = self._curated_path()
        ma = self._mask_autosave_path()

        @thread_worker
        def _work():
            save_image(out, arr, meta)
            # original now holds the curated result -> remove obsolete sidecars
            for side in (curated, ma):
                try:
                    if side != out and side.is_file():
                        side.unlink()
                except Exception:
                    pass
            return out.name

        def _ok(name):
            self.path = self.base_path
            self._log(f"[SAVE] save completed -> {name} "
                      f"(original overwritten; sidecars removed)")

        def _err(exc):
            self._dirty = True  # commit failed -> keep marked so it can be re-saved
            self._log(f"[SAVE][ERROR] {type(exc).__name__}: {exc}; "
                      f"original + sidecars left intact.")

        worker = _work()
        worker.returned.connect(_ok)
        worker.errored.connect(_err)
        worker.finished.connect(self._on_save_finished)
        self._save_worker = worker
        self._save_running = True
        worker.start()
        self._log(f"[SAVE] writing full 3-channel OVER original -> {out.name} "
                  f"(this can take a while)...")

    def _on_close(self, event):
        """On close: if there are unsaved mask edits, write the full 3-channel
        image SYNCHRONOUSLY to the non-destructive _CURATED safety net (so the
        process cannot exit mid-write and the original is untouched). If nothing
        changed since the last save, skip the write entirely."""
        if self._autosave_timer is not None:
            self._autosave_timer.stop()

        if not self._dirty:
            self._log("[CLOSE] no unsaved changes; nothing to save.")
            self._orig_close(event)
            return

        if self.mask_layer is not None and self.arr is not None:
            try:
                self._sync_mask_into_arr()
                out = self._curated_path()
                self._log(f"[CLOSE] saving progress -> {out.name} ...")
                save_image(out, self.arr, self.meta)
                self._dirty = False
                self._log(f"[CLOSE] save completed -> {out.name}")
            except Exception as e:
                self._log(f"[CLOSE][ERROR] {type(e).__name__}: {e}; "
                          f"mask autosave kept.")
        self._orig_close(event)


In [6]:
# Default folder the file explorer opens in (you can change to wherever your images live, or just navigate to the folder each time from the GUI).
masks_folder = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate")

In [7]:
curator = MaskCurator(default_folder=masks_folder)
curator.start()

In [8]:
input_path = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate\day5_6mm_dish1_device7_Merged_day5_6mm_dish1_device7_Merged.tif")
image = tifffile.imread(input_path)
print(image.shape)

FileNotFoundError: [Errno 2] No such file or directory: '\\\\haase.embl.es\\haase\\Bel\\IMAGES_FOR_VASCUMAP_RETRAINING\\fluorescent_cells_tifs\\3_channel_images_to_curate\\day5_6mm_dish1_device7_Merged_day5_6mm_dish1_device7_Merged.tif'

In [ ]:
for z in range(18, image.shape[0]):
    image[z,2,:,:] = image[17,2,:,:]

[<Image layer 'Image' at 0x1cfd4addde0>,
 <Image layer 'Image [1]' at 0x1d42f017ac0>,
 <Image layer 'Image [2]' at 0x1cf86293850>]

In [ ]:
for z in range(0, 19):
    image[z,2,:,:] = image[13,2,:,:]

In [ ]:
viewer = napari.Viewer()
viewer.add_image(image, channel_axis = 1)

In [ ]:
# Overwrite the input image as a 3D (ZCYX) stack with embedded pixel-size metadata.
# Uses safe_imwrite: writes to a temp file, verifies it, then atomically replaces
# input_path. If the write fails (e.g. wrong dtype) the original is left intact.
PX_XY = 2.0   # micron per pixel in X and Y
Z_SPACING = 5.0  # micron between z-slices

safe_imwrite(
    input_path,
    image,
    imagej=True,
    resolution=(1.0 / PX_XY, 1.0 / PX_XY),
    metadata={"axes": "ZCYX", "spacing": Z_SPACING, "unit": "micron"},
)
print(f"Saved {image.shape} stack to {input_path}")


Saved (46, 3, 7800, 5172) stack to Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate\day5_6mm_dish1_device7_Merged_day5_6mm_dish1_device7_Merged.tif


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\tifffile\tifffile.py:3821: UserWarning: <tifffile.TiffWriter 'day5_6mm_dish1_…vice7_Merged.tif'> truncating ImageJ file
  warnings.warn(
